### Loading the Dataset
#### Dataset Summary
Emotion is a dataset of English Twitter messages with six basic emotions: anger, fear, joy, love, sadness, and surprise. For more detailed information please refer to the paper.



In [ ]:
from datasets import load_dataset

emotion = load_dataset('emotion')
emotion.set_format(type='pandas')

In [ ]:
dataset=emotion

In [ ]:
dataset

In [ ]:
dataset['train'][:]

In [ ]:
classes = dataset['train'].features['label'].names
classes

In [ ]:
dataset['train']

### Data Analysis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re

print('Train shape:', (len(dataset['train']), len(dataset['train'].column_names)))
print('Validation shape:', (len(dataset['validation']), len(dataset['validation'].column_names)))
print('Test shape:', (len(dataset['test']), len(dataset['test'].column_names)))

pd.DataFrame(dataset['train'][:5])

In [ ]:
# Class distribution in train split
class_counts = pd.Series(dataset['train']['label']).value_counts().sort_values(ascending=False)
display(class_counts.to_frame('count'))

plt.figure(figsize=(8, 4))
class_counts.plot(kind='bar', color='#2E86AB')
plt.title('Class Distribution (Train Split)')
plt.xlabel('Emotion')
plt.ylabel('Number of Samples')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
train_word_count = pd.DataFrame({
    'label_name': dataset['train']['label'],
    'Words Per Tweet': [len(str(t).split()) for t in dataset['train']['text']]
})

train_word_count.boxplot('Words Per Tweet', by='label_name', figsize=(10, 5))
plt.title('Words Per Tweet by Emotion (Train Split)')
plt.suptitle('')
plt.xlabel('Emotion')
plt.ylabel('Words Per Tweet')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

### Text Preprocessing

Use exactly the same preprocessing pipeline for all splits so the model sees text in the same format during:

Training (train)
Tuning/evaluation (validation)
Final evaluation or inference (test)

In [ ]:
# import re

# # Expand common contractions first, then keep only lowercase words.
# contraction_map = {
#     "isn't": "is not", "isnt": "is not",
#     "didn't": "did not", "didnt": "did not",
#     "don't": "do not", "dont": "do not",
#     "can't": "can not", "cant": "can not",
#     "won't": "will not", "wont": "will not",
#     "i'm": "i am", "im": "i am",
#     "you're": "you are", "youre": "you are",
#     "it's": "it is", "its": "it is",
#     "that's": "that is", "thats": "that is",
#     "there's": "there is", "theres": "there is",
#     "we're": "we are", "were": "we are",
#     "they're": "they are", "theyre": "they are",
#     "i've": "i have", "ive": "i have",
#     "we've": "we have", "weve": "we have",
#     "they've": "they have", "theyve": "they have",
#     "i'll": "i will", "ill": "i will",
#     "you'll": "you will", "youll": "you will",
#     "we'll": "we will", "well": "we will",
#     "they'll": "they will", "theyll": "they will",
#     "i'd": "i would", "id": "i would",
#     "you'd": "you would", "youd": "you would",
#     "we'd": "we would", "wed": "we would",
#     "they'd": "they would", "theyd": "they would"
# }

# def preprocess_text(text):
#     text = str(text).lower().strip()

#     # Replace contractions as full-word matches only.
#     for short_form, full_form in contraction_map.items():
#         text = re.sub(rf"\b{re.escape(short_form)}\b", full_form, text)

#     # Keep only alphabetic words and spaces.
#     text = re.sub(r"[^a-z\s]", " ", text)
#     text = re.sub(r"\s+", " ", text).strip()

#     return text

# # Apply preprocessing directly to the text column.
# dataset = dataset.map(lambda x: {'text': preprocess_text(x['text']),"label": x["label"]})
# print(dataset['train'][:10])

### Tokenization

In [ ]:
from transformers import AutoTokenizer
model_ckpt = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

In [ ]:
text = "I love Machine Learning!. Tokenization is awesome"
encoded_text = tokenizer(text)
print(encoded_text)

In [ ]:

tokens = tokenizer.convert_ids_to_tokens(encoded_text.input_ids)
print(tokens)


In [ ]:

tokenizer.vocab_size, tokenizer.model_max_length


### Tokenization of Emotion Dataset

In [ ]:
dataset.reset_format()

In [ ]:
def tokenize(batch):
    # Shorter sequence length speeds up CPU training a lot
    return tokenizer(batch['text'], padding=True, truncation=True)

df_encoded = dataset.map(tokenize, batched=True,batch_size=None)


In [ ]:
print(tokenize(dataset['train'][:2]))

In [ ]:
df_encoded

### Model Building

In [ ]:
import torch
from transformers import AutoModelForSequenceClassification

num_labels = len(classes)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AutoModelForSequenceClassification.from_pretrained(model_ckpt, num_labels = num_labels).to(device)


### training

In [ ]:
from transformers import TrainingArguments

# CPU-optimized fast training config
batch_size = 64
model_name = "distilbert-finetuned-emotion-fast"

training_args = TrainingArguments(
    output_dir = model_name,
    num_train_epochs=2,
    learning_rate = 2e-5,
    per_device_train_batch_size= batch_size,
    per_device_eval_batch_size = batch_size,
    weight_decay=0.01,
    eval_strategy = 'epoch',
    disable_tqdm=False)

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(pred):
  labels = pred.label_ids
  preds = pred.predictions.argmax(-1)
  f1 = f1_score(labels, preds, average='weighted')
  acc = accuracy_score(labels, preds)
  return {"accuracy": acc, "f1": f1}


In [ ]:
from transformers import Trainer

# Use only a subset for faster CPU training
# train_n = 8000
# val_n = 800
# test_n = 800

# train_ds = df_encoded['train'].shuffle(seed=42).select(range(min(train_n, len(df_encoded['train']))))
# val_ds = df_encoded['validation'].shuffle(seed=42).select(range(min(val_n, len(df_encoded['validation']))))
# test_ds = df_encoded['test'].shuffle(seed=42).select(range(min(test_n, len(df_encoded['test']))))

# print('Using samples -> train:', len(train_ds), 'val:', len(val_ds), 'test:', len(test_ds))

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=df_encoded['train'],
    eval_dataset=df_encoded['validation'],
    compute_metrics=compute_metrics,
    processing_class=tokenizer
)

In [ ]:
trainer.train()

In [ ]:
preds_outputs = trainer.predict(df_encoded['test'])
preds_outputs.metrics

In [ ]:
import numpy as np
y_preds = np.argmax(preds_outputs.predictions, axis=1)
y_true = df_encoded['test'][:]['label']

In [ ]:
from sklearn.metrics import classification_report
print(classes)
print(classification_report(y_true, y_preds))

In [ ]:
# 5) Save once at the end (for Streamlit)
import os, json

artifact_dir = "distilbert-finetuned-emotion-fast"
os.makedirs(artifact_dir, exist_ok=True)

id2label = {i: label for i, label in enumerate(classes)}
label2id = {label: i for i, label in id2label.items()}
model.config.id2label = id2label
model.config.label2id = label2id

trainer.save_model(artifact_dir)
tokenizer.save_pretrained(artifact_dir)

with open(os.path.join(artifact_dir, "train_metrics.json"), "w", encoding="utf-8") as f:
    json.dump(train_output.metrics, f, indent=2)

In [ ]:
!jupyter nbconvert --clear-output --inplace Emotion_detection.ipynb
